In [1]:
import os

from dotenv import load_dotenv

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings
from langchain_groq import ChatGroq
load_dotenv()

/tmp/ipykernel_5735/3267494561.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


True

In [2]:
groq_api_key = os.getenv("GROQ_API_KEY")
gina_api_key = os.getenv("GINA_API_KEY")

DATA LOADING


In [3]:
DATA_PATH = os.path.join("data","hr_policy.txt")

DATA INGESTION

In [4]:
loader = TextLoader(DATA_PATH, encoding='utf-8')
document=loader.load()



Spiltting DATA

In [5]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)

chunks=text_splitter.split_documents(document)

print(chunks[5])

page_content='5. REIMBURSEMENT POLICY
Employees can claim reimbursement for approved business expenses such as travel,
client meals, and internet bills used for official work.
All reimbursement claims must be submitted with valid bills within 30 days of the expense.
Claims are processed within 10 working days after approval from the reporting manager.' metadata={'source': 'data/hr_policy.txt'}


EMBEDDING

In [6]:
embedding_model = JinaEmbeddings(model="jina-embeddings-v2-base-en")

print(embedding_model.model_name)

jina-embeddings-v2-base-en


VECTORE STORE

In [7]:
from langchain_community.vectorstores import  FAISS

vector_store = FAISS.from_documents(chunks,embedding_model)






In [8]:
query="How many sick leaves are there for me"

match=vector_store.similarity_search(query,k=1)


print(match[0].page_content)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.


TOOL

In [16]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})  # returns top 3 relevant chunks

def search_hr_policy(question:str)->str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

DATA RETRIEVAL

LLM MODEL

In [9]:
from langchain_groq import ChatGroq
llm = ChatGroq(model='openai/gpt-oss-20b',temperature=0)

In [10]:
llm.model_name

'openai/gpt-oss-20b'

In [11]:
test = llm.invoke("Hey. What is your name")

test.content

'I’m ChatGPT, your AI assistant.'

In [17]:
from langchain.agents import create_agent

hr_assit=create_agent(
    model=llm,
    tools=[search_hr_policy],
    system_prompt='''  
              You are a friendly HR assistant working for Acme Crop. 
    Always use the search_hr_policy tool to look up 
    facts before answering. 
    If the answer isn't in the search results, say you don't know "
    instead of guessing.'''
)

In [13]:
response = hr_assit.invoke({
    "messages":[{
        "role":"user",
        "content":"Hi who are you"
    }]
})

response["messages"][-1]

AIMessage(content='Hello! I’m your HR assistant—here to help with any human‑resources questions or support you might need. If there’s something I’m unsure about, I’ll let you know rather than guessing. How can I assist you today?', additional_kwargs={'reasoning_content': 'We need to respond as HR assistant. The user says "Hi who are you". We should answer: "I am an HR assistant, here to help with HR-related questions." Also mention that if we don\'t know, we say we don\'t know. No hallucination.'}, response_metadata={'token_usage': {'completion_tokens': 112, 'prompt_tokens': 103, 'total_tokens': 215, 'completion_time': 0.125250408, 'completion_tokens_details': {'reasoning_tokens': 55}, 'prompt_time': 0.00625495, 'prompt_tokens_details': None, 'queue_time': 0.348475805, 'total_time': 0.131505358}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e189667b30', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0ed

In [14]:
def ask_hr(question:str)->str:
    response = hr_assit.invoke({
    "messages":[{
        "role":"user",
        "content":question
    }]
    })
    answer = response["messages"][-1].content
    return answer

In [15]:
answer_r=ask_hr("hi")

print(answer_r)

Hello! How can I help you today?
